In [ ]:
import joblib
import numpy as np

# Load balanced datasets
X_train = joblib.load("X_train_balanced.pkl")
y_train = joblib.load("X_train_balanced.pkl")

X_val = joblib.load("X_val_balanced.pkl")
y_val = joblib.load("X_val_balanced.pkl")

X_test = joblib.load("X_test_balanced.pkl")
y_test = joblib.load("X_test_balanced.pkl")


# Load the tuple from pickle
data = joblib.load("X_train_balanced.pkl")

# Check what type of object it is
print(type(data))

# If it's a tuple or list, unpack it
if isinstance(data, (tuple, list)):
    X_train, y_train = data
    print("X_train shape:", X_train.shape)
    print("y_train shape:", y_train.shape)
else:
    # If it's just an array, then only features were saved
    X_train = data
    print("X_train shape:", X_train.shape)
    print("No labels found in this file.")

: 

Compute Class weights

In [2]:
from sklearn.utils.class_weight import compute_class_weight

cw = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = {i: cw[i] for i in range(len(cw))}
print("Class Weights:", class_weights)

Class Weights: {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0, 6: 1.0, 7: 1.0, 8: 1.0, 9: 1.0}


Explanation:
This ensures rare species are not ignored during training.


Build RESNET Model

In [3]:
import tensorflow as tf
print(tf.__version__)
from tensorflow import keras
print(keras.__name__)

AttributeError: module 'tensorflow' has no attribute '__version__'

In [4]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, applications, regularizers

IMG_SIZE = (128, 128)

# Base ResNet50
base_model = applications.ResNet50(
    include_top=False,
    input_shape=IMG_SIZE + (3,)
)
base_model.trainable = False  # freeze initially

# Custom classification head
model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dropout(0.4),
    layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

ImportError: cannot import name 'keras' from 'tensorflow' (unknown location)

Explanation:
ResNet50 extracts features, while the dense head learns diatom‑specific classification


Callbacks

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint("cnn_model_best.keras", save_best_only=True)
]

Training Phase1 (Frozen base)

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks,
    batch_size=32
)

Training Phase 2 (fine tuning)

In [ ]:
# Unfreeze base model
base_model.trainable = True

# Recompile with smaller LR
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    class_weight=class_weights,
    callbacks=callbacks,
    batch_size=32
)

Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

# Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_acc)

# Predictions
y_pred = np.argmax(model.predict(X_test), axis=1)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - ResNet50 CNN")
plt.show()

Savemodel

In [ ]:
import joblib

# Save model with joblib (wrap Keras model)
joblib.dump(model, "resnet50_cnn.pkl")
print("✅ ResNet50 CNN model saved as resnet50_cnn.pkl")